In [74]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [75]:
df = pd.read_csv('gurgaon_properties_feature_selectionV2.csv').drop(columns=['store room','floor_category','balcony'])

In [76]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,house,sector 11,4.00,5,4,Moderately Old,2025.0,0,1,Low
1,flat,sector 110,3.30,4,5,Relatively New,3032.0,1,1,Medium
2,flat,sector 104,2.30,3,4,Moderately Old,2217.0,1,1,Medium
3,house,sector 50,3.99,4,5,New Property,4500.0,1,0,Medium
4,flat,sector 37,1.45,3,3,Relatively New,1711.0,0,1,Medium


In [77]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished

In [78]:
# Numerical = bedRoom, bathroom, built_up_area, servant room
# Ordinal = property_type, furnishing_type, luxury_category 
# OHE = sector, agePossession

In [79]:
df['agePossession'] = df['agePossession'].replace(
    {
        'Relatively New':'new',
        'Moderately Old':'old',
        'New Property' : 'new',
        'Old Property' : 'old',
        'Under Construction' : 'under construction'
    }
)

In [80]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,house,sector 11,4.00,5,4,old,2025.0,0,1,Low
1,flat,sector 110,3.30,4,5,new,3032.0,1,1,Medium
2,flat,sector 104,2.30,3,4,old,2217.0,1,1,Medium
3,house,sector 50,3.99,4,5,new,4500.0,1,0,Medium
4,flat,sector 37,1.45,3,3,new,1711.0,0,1,Medium


In [81]:
df['property_type'] = df['property_type'].replace({'flat':0,'house':1})

C:\Users\kushs\AppData\Local\Temp\ipykernel_45620\71934247.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['property_type'] = df['property_type'].replace({'flat':0,'house':1})


In [82]:
df.head()

,property_type,sector,price,bedRoom,bathroom,agePossession,built_up_area,servant room,furnishing_type,luxury_category
0,1,sector 11,4.00,5,4,old,2025.0,0,1,Low
1,0,sector 110,3.30,4,5,new,3032.0,1,1,Medium
2,0,sector 104,2.30,3,4,old,2217.0,1,1,Medium
3,1,sector 50,3.99,4,5,new,4500.0,1,0,Medium
4,0,sector 37,1.45,3,3,new,1711.0,0,1,Medium


In [83]:
df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})

C:\Users\kushs\AppData\Local\Temp\ipykernel_45620\2576796079.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})


In [84]:
new_df = pd.get_dummies(df,columns=['sector','agePossession'],drop_first=True,dtype=int)

In [85]:
new_df

,property_type,price,bedRoom,bathroom,built_up_area,servant room,furnishing_type,luxury_category,sector_sector 10,sector_sector 102,sector_sector 103,sector_sector 104,sector_sector 105,sector_sector 106,sector_sector 107,sector_sector 108,sector_sector 109,sector_sector 11,sector_sector 110,sector_sector 111,sector_sector 112,sector_sector 113,sector_sector 12,sector_sector 13,sector_sector 14,sector_sector 15,sector_sector 17,sector_sector 2,sector_sector 21,sector_sector 22,sector_sector 23,sector_sector 24,sector_sector 25,sector_sector 26,sector_sector 27,sector_sector 28,sector_sector 3,sector_sector 30,sector_sector 31,sector_sector 33,sector_sector 36,sector_sector 37,sector_sector 38,sector_sector 39,sector_sector 4,sector_sector 40,sector_sector 41,sector_sector 42,sector_sector 43,sector_sector 45,sector_sector 46,sector_sector 47,sector_sector 48,sector_sector 49,sector_sector 5,sector_sector 50,sector_sector 51,sector_sector 52,sector_sector 53,sector_sector 54,sector_sector 55,sector_sector 56,sector_sector 57,sector_sector 58,sector_sector 59,sector_sector 6,sector_sector 60,sector_sector 61,sector_sector 62,sector_sector 63,sector_sector 65,sector_sector 66,sector_sector 67,sector_sector 68,sector_sector 69,sector_sector 7,sector_sector 70,sector_sector 71,sector_sector 72,sector_sector 73,sector_sector 74,sector_sector 76,sector_sector 77,sector_sector 78,sector_sector 79,sector_sector 8,sector_sector 80,sector_sector 81,sector_sector 82,sector_sector 83,sector_sector 84,sector_sector 85,sector_sector 86,sector_sector 88,sector_sector 89,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sohna,agePossession_old,agePossession_under construction
0,1,4.00,5,4,2025.0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,0,3.30,4,5,3032.0,1,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,2.30,3,4,2217.0,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,1,3.99,4,5,4500.0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,1.45,3,3,1711.0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3619,0,1.40,3,3,1735.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3620,0,1.00,3,3,1163.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3621,0,0.79,4,4,1765.0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3622,0,0.60,2,1,667.0,0,1,0,0,0,0,0,0,0,0,0,

In [86]:
X = new_df.drop(columns=['price'])
y = new_df['price']

In [87]:
y_log = np.log1p(y)

In [88]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [89]:
X_scaled = pd.DataFrame(X_scaled,columns=X.columns)

In [90]:
X_scaled

,property_type,bedRoom,bathroom,built_up_area,servant room,furnishing_type,luxury_category,sector_sector 10,sector_sector 102,sector_sector 103,sector_sector 104,sector_sector 105,sector_sector 106,sector_sector 107,sector_sector 108,sector_sector 109,sector_sector 11,sector_sector 110,sector_sector 111,sector_sector 112,sector_sector 113,sector_sector 12,sector_sector 13,sector_sector 14,sector_sector 15,sector_sector 17,sector_sector 2,sector_sector 21,sector_sector 22,sector_sector 23,sector_sector 24,sector_sector 25,sector_sector 26,sector_sector 27,sector_sector 28,sector_sector 3,sector_sector 30,sector_sector 31,sector_sector 33,sector_sector 36,sector_sector 37,sector_sector 38,sector_sector 39,sector_sector 4,sector_sector 40,sector_sector 41,sector_sector 42,sector_sector 43,sector_sector 45,sector_sector 46,sector_sector 47,sector_sector 48,sector_sector 49,sector_sector 5,sector_sector 50,sector_sector 51,sector_sector 52,sector_sector 53,sector_sector 54,sector_sector 55,sector_sector 56,sector_sector 57,sector_sector 58,sector_sector 59,sector_sector 6,sector_sector 60,sector_sector 61,sector_sector 62,sector_sector 63,sector_sector 65,sector_sector 66,sector_sector 67,sector_sector 68,sector_sector 69,sector_sector 7,sector_sector 70,sector_sector 71,sector_sector 72,sector_sector 73,sector_sector 74,sector_sector 76,sector_sector 77,sector_sector 78,sector_sector 79,sector_sector 8,sector_sector 80,sector_sector 81,sector_sector 82,sector_sector 83,sector_sector 84,sector_sector 85,sector_sector 86,sector_sector 88,sector_sector 89,sector_sector 9,sector_sector 90,sector_sector 91,sector_sector 92,sector_sector 93,sector_sector 95,sector_sector 99,sector_sohna,agePossession_old,agePossession_under construction
0,1.900181,1.530684,0.503752,0.133029,-0.741202,0.430135,-0.972470,-0.047036,-0.181844,-0.108283,-0.137245,-0.078152,-0.105644,-0.128646,-0.127533,-0.154978,16.057931,-0.083345,-0.08165,-0.079919,-0.10958,-0.095863,-0.03717,-0.076344,-0.052602,-0.049896,-0.076344,-0.049896,-0.074494,-0.141362,-0.08165,-0.102941,-0.106972,-0.023499,-0.101563,-0.070652,-0.03717,-0.052602,-0.138285,-0.078152,-0.184259,-0.052602,-0.055178,-0.113386,-0.040723,-0.055178,-0.028784,-0.114628,-0.049896,-0.055178,-0.066593,-0.139317,-0.115857,-0.097318,-0.131932,-0.049896,-0.062275,-0.086639,-0.085007,-0.052602,-0.126411,-0.083345,-0.049896,-0.052602,-0.055178,-0.064469,-0.106972,-0.066593,-0.089815,-0.158672,-0.106972,-0.138285,-0.091363,-0.162290,-0.101563,-0.171035,-0.086639,-0.089815,-0.028784,-0.070652,-0.064469,-0.086639,-0.066593,-0.151202,-0.043992,-0.03717,-0.156835,-0.12975,-0.138285,-0.117074,-0.175262,-0.131932,-0.083345,-0.127533,-0.079919,-0.158672,-0.068652,-0.168454,-0.049896,-0.126411,-0.108283,-0.209234,1.625488,-0.292156
1,-0.526266,0.728718,1.190324,0.955285,1.349160,0.430135,0.455501,-0.047036,-0.181844,-0.108283,-0.137245,-0.078152,-0.105644,-0.128646,-0.127533,-0.154978,-0.062275,11.998333,-0.08165,-0.079919,-0.10958,-0.095863,-0.03717,-0.076344,-0.052602,-0.049896,-0.076344,-0.049896,-0.074494,-0.141362,-0.08165,-0.102941,-0.106972,-0.023499,-0.101563,-0.070652,-0.03717,-0.052602,-0.138285,-0.078152,-0.184259,-0.052602,-0.055178,-0.113386,-0.040723,-0.055178,-0.028784,-0.114628,-0.049896,-0.055178,-0.066593,-0.139317,-0.115857,-0.097318,-0.131932,-0.049896,-0.062275,-0.086639,-0.085007,-0.052602,-0.126411,-0.083345,-0.049896,-0.052602,-0.055178,-0.064469,-0.106972,-0.066593,-0.089815,-0.158672,-0.106972,-0.138285,-0.091363,-0.162290,-0.101563,-0.171035,-0.086639,-0.089815,-0.028784,-0.070652,-0.064469,-0.086639,-0.066593,-0.151202,-0.043992,-0.03717,-0.156835,-0.12975,-0.138285,-0.117074,-0.175262,-0.131932,-0.083345,-0.127533,-0.079919,-0.158672,-0.068652,-0.168454,-0.049896,-0.126411,-0.108283,-0.209234,-0.615200,-0.292156
2,-0.526266,-0.073248,0.503752,0.289805,1.349160,0.430135,0.455501,-0.047036,-0.181844,-0.108283,7.286258,-0.078152,-0.105644,-0.128646,-0.127533,-0.154978,-0.062275,-0.083345,-0.08165,

In [91]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')

In [92]:
scores.mean(),scores.std()

(np.float64(0.8412545275240835), np.float64(0.022407727071017423))

In [93]:
lr = LinearRegression()
ridge = Ridge(alpha=0.0001)

In [94]:
lr.fit(X_scaled,y_log)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [99]:
lr.coef_

array([ 0.11279299,  0.05854052,  0.06068365,  0.22080779,  0.05588357,
       -0.00452731,  0.01176143,  0.01453063,  0.06287553,  0.02023244,
        0.03043628, -0.00245028,  0.0211045 ,  0.01521788,  0.04345817,
        0.0566441 ,  0.00539546,  0.02532403,  0.03116244,  0.04653087,
        0.05437416,  0.00485652, -0.00208371,  0.04204238,  0.02138374,
        0.03000331,  0.02467778,  0.00197409,  0.05223129,  0.03953169,
        0.0385562 ,  0.08017538,  0.09772959,  0.0124076 ,  0.06611888,
        0.00618347,  0.01467271,  0.03465113,  0.05520514,  0.02018261,
        0.03134026,  0.02474229,  0.01551209,  0.00665694,  0.02221255,
        0.02920692,  0.01582799,  0.07510043,  0.03731337,  0.02899267,
        0.02801179,  0.07226738,  0.05784098,  0.00439208,  0.09893836,
        0.0061648 ,  0.01839796,  0.0789873 ,  0.06377814,  0.02518367,
        0.03812816,  0.0437217 ,  0.03272503,  0.03346204, -0.00223223,
        0.03690147,  0.05933023,  0.05615406,  0.05917906,  0.10

In [95]:
ridge.fit(X_scaled,y_log)

,alpha,0.0001
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [100]:
ridge.coef_

array([ 0.11279299,  0.05854052,  0.06068365,  0.22080778,  0.05588356,
       -0.00452731,  0.01176144,  0.01453059,  0.06287537,  0.02023234,
        0.03043615, -0.00245035,  0.02110441,  0.01521776,  0.04345806,
        0.05664396,  0.00539541,  0.02532396,  0.03116236,  0.0465308 ,
        0.05437406,  0.00485644, -0.00208374,  0.04204231,  0.02138369,
        0.03000327,  0.02467771,  0.00197404,  0.05223122,  0.03953157,
        0.03855613,  0.08017529,  0.0977295 ,  0.01240758,  0.06611879,
        0.00618341,  0.01467268,  0.03465109,  0.05520502,  0.02018254,
        0.0313401 ,  0.02474225,  0.01551204,  0.00665684,  0.02221251,
        0.02920687,  0.01582797,  0.07510033,  0.03731332,  0.02899262,
        0.02801173,  0.07226726,  0.05784088,  0.004392  ,  0.09893825,
        0.00616476,  0.01839791,  0.07898723,  0.06377806,  0.02518362,
        0.03812805,  0.04372163,  0.03272499,  0.033462  , -0.00223228,
        0.03690141,  0.05933014,  0.056154  ,  0.05917898,  0.10

In [101]:
coef_df = pd.DataFrame(lr.coef_.reshape(1,104),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [102]:
coef_df

,feature,coef
0,property_type,0.112793
1,bedRoom,0.058541
2,bathroom,0.060684
3,built_up_area,0.220808
4,servant room,0.055884
...,...,...
99,sector_sector 95,0.002455
100,sector_sector 99,0.013141
101,sector_sohna,0.017164
102,agePossession_old,-0.008119


In [103]:
# 1. Import necessary libraries
import statsmodels.api as sm

# 2. Add a constant to X
X_with_const = sm.add_constant(X_scaled)

# 3. Fit the model
model = sm.OLS(y_log, X_with_const).fit()

# 4. Obtain summary statistics
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.855
Model:                            OLS   Adj. R-squared:                  0.850
Method:                 Least Squares   F-statistic:                     198.9
Date:                Tue, 21 Jul 2026   Prob (F-statistic):               0.00
Time:                        02:19:23   Log-Likelihood:                 460.85
No. Observations:                3624   AIC:                            -711.7
Df Residuals:                    3519   BIC:                            -61.19
Df Model:                         104                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
const   

In [104]:
y_log.std()

0.558894978881325

In [105]:
X_scaled['bedRoom'].std()

1.0001379976547116

In [106]:
0.21 * (0.557/1)

0.11697

In [107]:
np.expm1(0.030)

np.float64(0.030454533953516858)